<!-- Opcional: abra este notebook no Colab se o repositório estiver publicado no GitHub -->
<!-- <a target="_blank" href="...E02_Ensemble_Learning.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> -->

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação do sklearn.

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:
- correto
- reprodutível
- bem estruturado
- criticamente analisado

# Dicas importantes

## Sobre o dataset (Fashion MNIST)

- Utilize `fetch_openml` do sklearn para carregar os dados
- Use: `as_frame=False`
- Use o nome `Fashion-MNIST` (versão 1 no OpenML), com `parser='auto'` se necessário
- Converta os rótulos para inteiro:
  
  ```python
  y = y.astype(int)
  ```

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST`
Realize a separação em treino e teste
Utilize `train_test_split` com controle de aleatoriedade
Retorne: `X_train`, `X_test`, `y_train`, `y_test`

Depois responda: 
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

### Normalização e modelos baseados em árvores

Para **Random Forest** e **AdaBoost** (com ávores como aprendizes fracos), a normalização **não é obrigatória**: partições em eixos são invariantes a transformações monótonas por feature, e a escala absoluta dos pixels já é interpretável. Ainda assim, **escalar** (por exemplo, dividir por 255 ou usar `StandardScaler` em um `Pipeline`) pode alterar ligeiramente as escolhas de divisão por causa de efeitos numéricos e do `min_samples_*`, e é útil se misturarmos com outros pré-processamentos. Aqui mantemos os pixels em escala original após conversão numérica, e usamos `StandardScaler` apenas nos experimentos com `Pipeline` para comparar.

In [1]:
"""Preparação dos dados: Fashion-MNIST (OpenML), split estratificado e tipos numéricos."""

import sys

import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split


def load_data(seed=42, test_size=0.2):

    # fetch_openml baixa o conjunto Fashion-MNIST (10 classes, 784 atributos).
    data = fetch_openml(name="Fashion-MNIST", version=1, as_frame=False, parser="auto")
    X = np.asarray(data.data, dtype=np.float64)
    y = np.asarray(data.target).astype(int)

    # Estratificação preserva a proporção das classes em treino e teste.
    return train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=seed,
        stratify=y,
    )


# Durante pytest o módulo é importado: não baixar dados nem imprimir aqui.
if "pytest" not in sys.modules:
    _Xt, _Xs, _yt, _ys = load_data(seed=42)
    print("Shapes:", _Xt.shape, _Xs.shape, _yt.shape, _ys.shape)

Shapes: (56000, 784) (14000, 784) (56000,) (14000,)


# Questão 2

Implemente as funções:

`train_random_forest(X_train, y_train, seed)`
`train_adaboost(X_train, y_train, seed)`

## Requisitos:

Utilizar os modelos do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [ ]:
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier


def train_random_forest(X_train, y_train, seed=42, **kwargs):
    """Treina RandomForestClassifier com random_state fixo (reprodutibilidade)."""
    params = dict(n_estimators=200, random_state=seed, n_jobs=-1)
    params.update(kwargs)
    model = RandomForestClassifier(**params)
    model.fit(X_train, y_train)
    return model


def train_adaboost(X_train, y_train, seed=42, **kwargs):
    """Treina AdaBoostClassifier multiclasse (SAMME) com random_state fixo."""
    params = dict(n_estimators=150, random_state=seed, algorithm="SAMME")
    params.update(kwargs)
    model = AdaBoostClassifier(**params)
    model.fit(X_train, y_train)
    return model


if "pytest" not in sys.modules:
    _demo_rf = train_random_forest(_Xt, _yt, seed=42)
    _demo_ab = train_adaboost(_Xt, _yt, seed=42)
    print("Modelos treinados:", type(_demo_rf).__name__, type(_demo_ab).__name__)

C:\Users\gustavo.carneiro\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(


# Questão 3

Implemente a função:

- `evaluate(model, X_test, y_test)`

Ela deve:
- Realizar predições
- Retornar a acurácia do modelo

**Solução**:

In [ ]:
from sklearn.metrics import accuracy_score


def evaluate(model, X_test, y_test):
    """Prediz em X_test e retorna acurácia (contrato exigido pelo enunciado)."""
    y_pred = model.predict(X_test)
    return float(accuracy_score(y_test, y_pred))


def classification_metrics_dict(model, X, y, average="weighted"):
    """Métricas completas para relatório (precisão, recall, F1, acurácia)."""
    from sklearn.metrics import f1_score, precision_score, recall_score

    y_pred = model.predict(X)
    return {
        "accuracy": float(accuracy_score(y, y_pred)),
        "precision": float(precision_score(y, y_pred, average=average, zero_division=0)),
        "recall": float(recall_score(y, y_pred, average=average, zero_division=0)),
        "f1": float(f1_score(y, y_pred, average=average, zero_division=0)),
    }


if "pytest" not in sys.modules:
    print("Acurácia RF (teste):", evaluate(_demo_rf, _Xs, _ys))

A função `evaluate` devolve apenas a **acurácia**, como pedido na Questão 3, para manter o contrato usado em `run_pipeline` e em testes automatizados. As **demais métricas** usam `sklearn.metrics` na função auxiliar `classification_metrics_dict` nas questões seguintes.

# Questão 4

Implemente a função:

- `run_pipeline(model_type="rf", seed=42)`

Ela deve:
- Carregar os dados
- Treinar o modelo escolhido (`rf` ou `ab`)
- Avaliar o modelo
- Retornar a acurácia

**Solução**:

In [ ]:
def run_pipeline(model_type="rf", seed=42):
    
    X_train, X_test, y_train, y_test = load_data(seed=seed)
    if model_type == "rf":
        model = train_random_forest(X_train, y_train, seed=seed)
    elif model_type == "ab":
        model = train_adaboost(X_train, y_train, seed=seed)
    else:
        raise ValueError("model_type deve ser 'rf' ou 'ab'.")
    return evaluate(model, X_test, y_test)


if "pytest" not in sys.modules:
    print("Pipeline RF (seed=42):", run_pipeline("rf", seed=42))
    print("Pipeline AB (seed=42):", run_pipeline("ab", seed=42))

### Sobre overfitting em *ensembles* de árvores

Em **uma única árvore** muito profunda, o treino pode chegar perto de 100% por memorização. Em **Random Forest**, o agrégio de árvores com *bagging* e subamostragem de atributos **reduz variância**, mas a floresta ainda pode ter **gap treino–teste** se as árvores forem muito profundas ou se houver poucos dados. **AdaBoost** pode **aumentar o ajuste** ao focar em erros; com muitos estimadores fracos ou base muito complexa, o gap também cresce.

Por isso comparamos **acurácia (e outras métricas) no treino e no teste** na Questão 7: um desempenho muito melhor no treino é **sinal de overfitting**; desempenho baixo em ambos sugere **underfitting**.

# Questão 5

Execute o pipeline para ambos os modelos:

- Random Forest
- AdaBoost

## Apresente:
- Acurácia, Precisão, Recall e F1-Score de cada modelo

## Responda:
- Qual modelo apresentou melhor desempenho inicial?

**Solução**:

In [ ]:
if "pytest" not in sys.modules:
    import pandas as pd
    from sklearn.metrics import classification_report

    X_train, X_test, y_train, y_test = load_data(seed=42)
    rf_model = train_random_forest(X_train, y_train, seed=42)
    ab_model = train_adaboost(X_train, y_train, seed=42)

    rows = []
    for name, mdl in [("RandomForest", rf_model), ("AdaBoost", ab_model)]:
        rows.append({"modelo": name, **classification_metrics_dict(mdl, X_test, y_test)})

    df_cmp = pd.DataFrame(rows).set_index("modelo")
    print("Comparação no conjunto de teste (média ponderada das classes):")
    print(df_cmp.round(4).to_string())

    print("\nRelatório detalhado — Random Forest:")
    print(classification_report(y_test, rf_model.predict(X_test), digits=4))
    print("Relatório detalhado — AdaBoost:")
    print(classification_report(y_test, ab_model.predict(X_test), digits=4))

    best = df_cmp["f1"].idxmax()
    print(f"\nMelhor F1 (ponderado) inicial: {best} (F1={df_cmp.loc[best, 'f1']:.4f}).")

# Questão 6

Execute o pipeline utilizando diferentes seeds (ex: 42 e 7).

## Analise:
- Os resultados mudaram?

## Responda:
- O experimento é reprodutível? Justifique.

**Solução**:

In [ ]:
if "pytest" not in sys.modules:
    import pandas as pd

    seeds = [42, 7]
    records = []
    for s in seeds:
        records.append({"seed": s, "modelo": "RF", "accuracy_teste": run_pipeline("rf", seed=s)})
        records.append({"seed": s, "modelo": "AB", "accuracy_teste": run_pipeline("ab", seed=s)})

    df_seeds = pd.DataFrame(records)
    print("Acurácia no teste variando apenas random_state (split + modelo):")
    print(
        df_seeds.pivot(index="seed", columns="modelo", values="accuracy_teste")
        .round(4)
        .to_string()
    )

    print(
        "\nReprodutibilidade: com a MESMA seed, split e RNG do modelo são idênticos → "
        "mesmos resultados. Seeds diferentes alteram a partição treino/teste e a aleatoriedade "
        "intrínseca (RF/AB), então métricas podem variar levemente; isso não contradiz reprodutibilidade, "
        "que significa repetir o experimento com os mesmos hiperparâmetros e seed e obter o mesmo resultado."
    )

# Questão 7

Para pelo menos um dos modelos:

- Compare a acurácia em treino e teste

## Responda:
- Existe overfitting?
- Qual modelo tende a sofrer mais com isso?

In [ ]:
if "pytest" not in sys.modules:
    X_train, X_test, y_train, y_test = load_data(seed=42)
    for name, trainer in [("RandomForest", train_random_forest), ("AdaBoost", train_adaboost)]:
        m = trainer(X_train, y_train, seed=42)
        tr = classification_metrics_dict(m, X_train, y_train)
        te = classification_metrics_dict(m, X_test, y_test)
        print(f"\n{name} — treino vs teste (acurácia / F1 ponderado):")
        print(f"  treino: acc={tr['accuracy']:.4f}, F1={tr['f1']:.4f}")
        print(f"  teste:  acc={te['accuracy']:.4f}, F1={te['f1']:.4f}")
        print(f"  gap acc (treino - teste): {tr['accuracy'] - te['accuracy']:.4f}")

    print(
        "\nInterpretação: Random Forest costuma encaixar muito bem o treino (árvores profundas); "
        "o gap maior indica mais risco de overfitting relativo. AdaBoost (stumps) frequentemente "
        "apresenta gap menor, mas pode underfitting se poucos estimadores."
    )

# Questão 8

Varie pelo menos um hiperparâmetro em cada modelo:

- Random Forest: `n_estimators`
- AdaBoost: `n_estimators`

## Analise:
- O desempenho muda significativamente?

## Responda:
- Qual modelo é mais sensível a mudanças?

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def make_rf_pipeline(n_estimators, seed, with_scaler=False):
    steps = []
    if with_scaler:
        steps.append(("scaler", StandardScaler()))
    steps.append(
        (
            "clf",
            RandomForestClassifier(
                n_estimators=n_estimators, random_state=seed, n_jobs=-1
            ),
        )
    )
    return Pipeline(steps)


def make_ab_pipeline(n_estimators, seed, with_scaler=False):
    steps = []
    if with_scaler:
        steps.append(("scaler", StandardScaler()))
    steps.append(
        (
            "clf",
            AdaBoostClassifier(
                n_estimators=n_estimators, random_state=seed, algorithm="SAMME"
            ),
        )
    )
    return Pipeline(steps)


if "pytest" not in sys.modules:
    import pandas as pd

    X_train, X_test, y_train, y_test = load_data(seed=42)

    exp_rows = []
    for n in [50, 100, 200]:
        for scale in [False, True]:
            pipe = make_rf_pipeline(n, seed=42, with_scaler=scale)
            pipe.fit(X_train, y_train)
            exp_rows.append(
                {
                    "modelo": "RF",
                    "n_estimators": n,
                    "StandardScaler": scale,
                    "acc_teste": float(accuracy_score(y_test, pipe.predict(X_test))),
                }
            )

    for n in [50, 100, 200]:
        for scale in [False, True]:
            pipe = make_ab_pipeline(n, seed=42, with_scaler=scale)
            pipe.fit(X_train, y_train)
            exp_rows.append(
                {
                    "modelo": "AB",
                    "n_estimators": n,
                    "StandardScaler": scale,
                    "acc_teste": float(accuracy_score(y_test, pipe.predict(X_test))),
                }
            )

    df_exp = pd.DataFrame(exp_rows)
    print("Experimentos com Pipeline (seed=42 fixo no split e nos modelos):")
    print(
        df_exp.sort_values(["modelo", "n_estimators", "StandardScaler"])
        .reset_index(drop=True)
        .to_string()
    )

    rng = np.random.default_rng(42)
    extra_seeds = [int(x) for x in rng.integers(0, 10_000, size=3)]
    print("\nMesmo pipeline RF (n_estimators=100, sem scaler), seeds extras:")
    for s in extra_seeds:
        Xtr, Xte, ytr, yte = load_data(seed=s)
        p = make_rf_pipeline(100, seed=s, with_scaler=False)
        p.fit(Xtr, ytr)
        print(f"  seed={s}: acc_teste={accuracy_score(yte, p.predict(Xte)):.4f}")

    print(
        "\nSensibilidade: RF em geral estabiliza com mais árvores; AB pode oscilar mais com poucos "
        "estimadores e com SAMME em multiclasse. O scaler costuma ter impacto pequeno em árvores, "
        "mas aparece na tabela para confrontar com o argumento da Questão 1."
    )

# Questão 9

Responda (máx. 2 parágrafos por item):

1. A acurácia é suficiente para avaliar os modelos?
2. Como você garante que o resultado não ocorreu por acaso?
3. Cite dois possíveis problemas metodológicos neste experimento.
4. O pipeline implementado é confiável? Justifique.

### Reflexão crítica

**1. A acurácia é suficiente?**  
Não, em geral. Com classes balanceadas ela resume bem o acerto global, mas mascara erros desiguais entre categorias (por exemplo, confundir calçado com bolsa). Por isso usamos **precisão, recall e F1** com média ponderada e o `classification_report` por classe: em problemas custo-assimétricos ou com raros positivos, métricas por classe e matrizes de confusão são ainda mais importantes.

**2. Como garantir que o resultado não foi “por acaso”?**  
Fixamos `random_state` no split e nos modelos para **reprodutibilidade** (mesmo script → mesmos números), o que não prova significância estatística. Para ir além, seriam necessários **validação cruzada**, intervalos de confiança (bootstrap), testes ou comparação contra um *baseline* forte; repetir com várias seeds mostra **variabilidade** esperada, não aleatoriedade “sem causa”.

**3. Dois problemas metodológicos**  
(i) **Um único holdout** treino/teste: a estimativa de desempenho tem alta variância e depende da sorte do corte. (ii) **Ajuste implícito de hiperparâmetros** olhando o teste várias vezes (varrer `n_estimators` e escolher pelo teste) **infla otimisticamente** a métrica; o correto seria validação interna (CV) e um conjunto de validação ou nested CV.

**4. O pipeline é confiável?**  
É **consistente e auditável**: funções explícitas, `Pipeline` nos experimentos e seeds fixas. As limitações são de **escopo**: sem busca formal de hiperparâmetros com CV, sem análise de calibração nem de erros sistemáticos por classe, e dependência da versão do OpenML/sklearn. Melhorias naturais: **GridSearchCV/RandomizedSearchCV**, validação cruzada estratificada, registrar versões de bibliotecas e, se necessário, subamostrar para tempo de execução em CI mantendo estratificação.